#init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col

#Reading from the bronze table

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "customer_firstname",
    "cst_lastname": "customer_lastname",
    "cst_marital_status": "customer_marital_status",
    "cst_gndr": "customer_gender",
    "cst_create_date": "customer_create_date"
}

In [0]:
df=(
    spark.table("workspace.bronze.crm_cust_info")
)

#Data transformation

In [0]:
df.display()

##Trimming

In [0]:
for field in df.schema.fields:
   if isinstance(field.dataType,StringType):
      df=df.withColumn(field.name,trim(col(field.name)))


##Normalisation

In [0]:
df=(
    df
    .withColumn(
    "cst_marital_status",
    F.when(F.upper(F.col("cst_marital_status"))=="M","Married")
    .when(F.upper(F.col("cst_marital_status"))=="S","Single")
    .otherwise ("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr"))=="M","Male")
        .when(F.upper(F.col("cst_gndr"))=="F","Female")
        .otherwise ("n/a")
    )
)

##Renaming the column


In [0]:
for old_name,new_name in RENAME_MAP.items():
    df=df.withColumnRenamed(old_name,new_name)

In [0]:
df.display()

#writing into silver table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("silver.crm_customer")
)

In [0]:
%sql
select * from workspace.silver.crm_customer